read Json data (batch mode)
here we are going to explore the volumes and understand how to flat the json files

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df=spark.read.format("json").option("inferSchema",True)\
.option("multiline",True).load("/Volumes/workspace/streaming/stream_volume")
display(df)

customer,items,metadata,order_id,payment,timestamp
"List(List(Toronto, Canada, M5H 2N2), 501, john@example.com, John Doe)","List(List(I100, 25.99, Wireless Mouse, 2), List(I101, 15.49, USB-C Adapter, 1))","List(List(campaign, back_to_school), List(channel, email))",ORD1001,"List(Credit Card, TXN7890)",2025-06-01T10:15:00Z


In [0]:
df=df.select("items","order_id","timestamp","customer.customer_id","customer.name","customer.email","customer.address.city","customer.address.country","customer.address.postal_code","payment","metadata")

df=df.withColumn("items", explode_outer("items"))

display(df)

items,order_id,timestamp,customer_id,name,email,city,country,postal_code,payment,metadata
"List(I100, 25.99, Wireless Mouse, 2)",ORD1001,2025-06-01T10:15:00Z,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,"List(Credit Card, TXN7890)","List(List(campaign, back_to_school), List(channel, email))"
"List(I101, 15.49, USB-C Adapter, 1)",ORD1001,2025-06-01T10:15:00Z,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,"List(Credit Card, TXN7890)","List(List(campaign, back_to_school), List(channel, email))"


In [0]:
df=df.select("items.item_id","items.price","items.product_name","items.quantity","order_id","timestamp","name","email","city","country","postal_code","payment","metadata")
display(df)

item_id,price,product_name,quantity,order_id,timestamp,name,email,city,country,postal_code,payment,metadata
I100,25.99,Wireless Mouse,2,ORD1001,2025-06-01T10:15:00Z,John Doe,john@example.com,Toronto,Canada,M5H 2N2,"List(Credit Card, TXN7890)","List(List(campaign, back_to_school), List(channel, email))"
I101,15.49,USB-C Adapter,1,ORD1001,2025-06-01T10:15:00Z,John Doe,john@example.com,Toronto,Canada,M5H 2N2,"List(Credit Card, TXN7890)","List(List(campaign, back_to_school), List(channel, email))"


In [0]:
df=df.select("item_id","price","product_name","quantity","order_id","timestamp","name","email","city","country","postal_code","payment.method","payment.transaction_id","metadata")
display(df)

item_id,price,product_name,quantity,order_id,timestamp,name,email,city,country,postal_code,method,transaction_id,metadata
I100,25.99,Wireless Mouse,2,ORD1001,2025-06-01T10:15:00Z,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,"List(List(campaign, back_to_school), List(channel, email))"
I101,15.49,USB-C Adapter,1,ORD1001,2025-06-01T10:15:00Z,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,"List(List(campaign, back_to_school), List(channel, email))"


In [0]:
df=df.select("item_id","price","product_name","quantity","order_id","timestamp","name","email","city","country","postal_code","method","transaction_id","metadata")


df=df.withColumn("metadata", explode_outer("metadata"))

display(df)

item_id,price,product_name,quantity,order_id,timestamp,name,email,city,country,postal_code,method,transaction_id,metadata
I100,25.99,Wireless Mouse,2,ORD1001,2025-06-01T10:15:00Z,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,"List(campaign, back_to_school)"
I100,25.99,Wireless Mouse,2,ORD1001,2025-06-01T10:15:00Z,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,"List(channel, email)"
I101,15.49,USB-C Adapter,1,ORD1001,2025-06-01T10:15:00Z,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,"List(campaign, back_to_school)"
I101,15.49,USB-C Adapter,1,ORD1001,2025-06-01T10:15:00Z,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,"List(channel, email)"


In [0]:
df=df.select("*","metadata.key","metadata.value").drop("metadata")
display(df)


item_id,price,product_name,quantity,order_id,timestamp,name,email,city,country,postal_code,method,transaction_id,key,value,key,value
I100,25.99,Wireless Mouse,2,ORD1001,2025-06-01T10:15:00Z,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,campaign,back_to_school,campaign,back_to_school
I100,25.99,Wireless Mouse,2,ORD1001,2025-06-01T10:15:00Z,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,channel,email,channel,email
I101,15.49,USB-C Adapter,1,ORD1001,2025-06-01T10:15:00Z,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,campaign,back_to_school,campaign,back_to_school
I101,15.49,USB-C Adapter,1,ORD1001,2025-06-01T10:15:00Z,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,channel,email,channel,email


In [0]:
display(df)


item_id,price,product_name,quantity,order_id,timestamp,name,email,city,country,postal_code,method,transaction_id,key,value,key,value
I100,25.99,Wireless Mouse,2,ORD1001,2025-06-01T10:15:00Z,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,campaign,back_to_school,campaign,back_to_school
I100,25.99,Wireless Mouse,2,ORD1001,2025-06-01T10:15:00Z,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,channel,email,channel,email
I101,15.49,USB-C Adapter,1,ORD1001,2025-06-01T10:15:00Z,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,campaign,back_to_school,campaign,back_to_school
I101,15.49,USB-C Adapter,1,ORD1001,2025-06-01T10:15:00Z,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,channel,email,channel,email


### read streaming data

now that we understand how to flat the json files, we are going to simulate the streaming of files applying the ONCE trigger

In [0]:
my_schema= """order_id STRING,
timestamp STRING,
customer STRUCT<
  customer_id: INT,
  name: STRING,
  email: STRING,
  address: STRUCT<
    city: STRING,
    postal_code: STRING,
    country: STRING
  >
>,
items ARRAY<STRUCT<
  item_id: STRING,
  product_name: STRING,
  quantity: INT,
  price: DOUBLE
>>,
payment STRUCT<
  method: STRING,
  transaction_id: STRING
>,
metadata ARRAY<STRUCT<
  key: STRING,
  value: STRING
>>"""


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

df=spark.readStream.format("json").option("multiline",True)\
.schema(my_schema)\
.load("/Volumes/workspace/streaming/stream_volume")

#transformations
df=df.select("items","order_id","timestamp","customer.customer_id","customer.name","customer.email","customer.address.city","customer.address.country","customer.address.postal_code","payment","metadata")

df=df.withColumn("items", explode_outer("items"))

df=df.select("items.item_id","items.price","items.product_name","items.quantity","order_id","timestamp","name","email","city","country","postal_code","payment.method","payment.transaction_id","metadata")

df=df.withColumn("metadata", explode_outer("metadata"))
df=df.select("*","metadata.key","metadata.value").drop("metadata")





In [0]:
df.writeStream.format("delta").outputMode("append")\
    .trigger(once=True)\
    .option("path","/Volumes/workspace/streaming/stream_volume/jsonsink/data")\
    .option("checkpointLocation", "/Volumes/workspace/streaming/stream_volume/jsonsink/checkpoint").start()

In [0]:
%sql
select * from delta.`/Volumes/workspace/streaming/stream_volume/jsonsink/data`

item_id,price,product_name,quantity,order_id,timestamp,name,email,city,country,postal_code,method,transaction_id,key,value
I100,25.99,Wireless Mouse,2,ORD1001,2025-06-01T10:15:00Z,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,campaign,back_to_school
I100,25.99,Wireless Mouse,2,ORD1001,2025-06-01T10:15:00Z,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,channel,email
I101,15.49,USB-C Adapter,1,ORD1001,2025-06-01T10:15:00Z,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,campaign,back_to_school
I101,15.49,USB-C Adapter,1,ORD1001,2025-06-01T10:15:00Z,John Doe,john@example.com,Toronto,Canada,M5H 2N2,Credit Card,TXN7890,channel,email
I102,45.0,Bluetooth Keyboard,1,ORD1002,2025-06-01T10:30:00Z,Alice Smith,alice@example.com,Vancouver,Canada,V5K 0A1,PayPal,TXN7891,campaign,cyber_monday
I102,45.0,Bluetooth Keyboard,1,ORD1002,2025-06-01T10:30:00Z,Alice Smith,alice@example.com,Vancouver,Canada,V5K 0A1,PayPal,TXN7891,channel,affiliate


### Archiving

In [0]:
dbutils.fs.mkdirs("/Volumes/workspace/streaming/stream_volume/jsonsource")

True

In [0]:
dbutils.fs.mkdirs("/Volumes/workspace/streaming/stream_volume/archive")

True

In [0]:
dbutils.fs.mkdirs("/Volumes/workspace/streaming/stream_volume/destination")

True

In [0]:
my_schema= """order_id STRING,
timestamp STRING,
customer STRUCT<
  customer_id: INT,
  name: STRING,
  email: STRING,
  address: STRUCT<
    city: STRING,
    postal_code: STRING,
    country: STRING
  >
>,
items ARRAY<STRUCT<
  item_id: STRING,
  product_name: STRING,
  quantity: INT,
  price: DOUBLE
>>,
payment STRUCT<
  method: STRING,
  transaction_id: STRING
>,
metadata ARRAY<STRUCT<
  key: STRING,
  value: STRING
>>"""


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


df=spark.readStream.format("json").option("multiline",True)\
.schema(my_schema)\
.option("cleanSource","archive")\
.option("sourceArchiveDir","/Volumes/workspace/streaming/stream_volume/archive")\
.load("/Volumes/workspace/streaming/stream_volume/jsonsource")

#transformations
df=df.select("items","order_id","timestamp","customer.customer_id","customer.name","customer.email","customer.address.city","customer.address.country","customer.address.postal_code","payment","metadata")

df=df.withColumn("items", explode_outer("items"))

df=df.select("items.item_id","items.price","items.product_name","items.quantity","order_id","timestamp","name","email","city","country","postal_code","payment.method","payment.transaction_id","metadata")

df=df.withColumn("metadata", explode_outer("metadata"))
df=df.select("*","metadata.key","metadata.value").drop("metadata")



In [0]:
df.writeStream.format("delta").outputMode("append")\
    .trigger(once=True)\
    .option("path","/Volumes/workspace/streaming/stream_volume/destination/data")\
    .option("checkpointLocation", "/Volumes/workspace/streaming/stream_volume/destination/checkpoint").start()